In [1]:
import pandas as pd

df = pd.read_csv(
    r"C:\data\complaints.csv",
    nrows=5
)

print(df.columns.tolist())

['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Consumer disputed?', 'Complaint ID']


In [35]:
print(df.columns.tolist())

['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Consumer disputed?', 'Complaint ID']


In [36]:
print(df.head())
print(df.columns.tolist())

  Date received                                            Product  \
0    2025-06-20  Credit reporting or other personal consumer re...   
1    2025-06-20                                    Debt collection   
2    2025-06-20  Credit reporting or other personal consumer re...   
3    2025-06-20  Credit reporting or other personal consumer re...   
4    2025-06-20  Credit reporting or other personal consumer re...   

               Sub-product                                 Issue  \
0         Credit reporting  Incorrect information on your report   
1  Telecommunications debt     Attempts to collect debt not owed   
2         Credit reporting           Improper use of your report   
3         Credit reporting           Improper use of your report   
4         Credit reporting  Incorrect information on your report   

                                       Sub-issue  \
0            Information belongs to someone else   
1                              Debt is not yours   
2  Reporting c

In [1]:
import pandas as pd

cols = pd.read_csv(
    r"C:\data\complaints.csv",
    nrows=0
)

print(cols.columns.tolist())

['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Consumer disputed?', 'Complaint ID']


In [2]:
import pandas as pd

sample = pd.read_csv(
    r"C:\data\complaints.csv",
    nrows=5
)

print(sample.columns)

Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Consumer consent provided?',
       'Submitted via', 'Date sent to company', 'Company response to consumer',
       'Timely response?', 'Consumer disputed?', 'Complaint ID'],
      dtype='str')


In [4]:
df = pd.read_csv(
    r"C:\data\complaints.csv",
    usecols=["Consumer complaint narrative"],
    nrows=10000
)

documents = (
    df["Consumer complaint narrative"]
    .fillna("")
    .tolist()
)

In [5]:
import pandas as pd

cols = pd.read_csv(
    r"C:\data\complaints.csv",
    nrows=0
)

print(cols.columns.tolist())

['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Consumer disputed?', 'Complaint ID']


In [6]:
import pandas as pd

df = pd.read_csv(
    r"C:\data\complaints.csv",
    usecols=["Consumer complaint narrative"],
    nrows=10000   # adjust if needed
)

df = df.dropna()

documents = df["Consumer complaint narrative"].tolist()

print(f"Loaded {len(documents)} complaints")

Loaded 2 complaints


In [7]:
import pandas as pd

df = pd.read_csv(
    r"C:\data\complaints.csv",
    usecols=["Consumer complaint narrative"],
    nrows=10000
)

df = df.dropna()

documents = df["Consumer complaint narrative"].tolist()

print(f"Loaded {len(documents)} documents")
print(documents[0][:200])

Loaded 2 documents
XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX Apt XXXX, XXXX, TX XXXX XXXX : XX/XX/XXXX TransUnion Consumer Solutions XXXX XXXX XXXX XXXX, PA XXXX XXXX : XXXX Re : Security Freeze Request Dear Sir/Madam, I,


In [8]:
print(type(documents))
print(len(documents))

<class 'list'>
2


In [9]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True,
    show_progress_bar=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [11]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    embeddings.astype(np.float32)
)

In [12]:
metadata = []

for doc in documents:
    metadata.append({
        "text": doc
    })

In [13]:
print("documents" in globals())

True


In [14]:
embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True,
    show_progress_bar=True
)

print(embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(2, 384)


In [15]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    embeddings.astype(np.float32)
)

print(index.ntotal)

2


In [16]:
metadata = [
    {"text": doc}
    for doc in documents
]

In [17]:
def retrieve_chunks(question, k=2):

    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    )

    distances, indices = index.search(
        query_embedding.astype(np.float32),
        k
    )

    return [
        metadata[idx]
        for idx in indices[0]
    ]

In [18]:
results = retrieve_chunks(
    "What are the common complaints?"
)

print(results[0]["text"][:300])

Subject : Dispute of Unauthorized Hard Inquiries on Credit Report XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX, Fl XXXX XX/XX/XXXX Social Security XXXX XX/XX/XXXX Consumer Financial Protection Bureau XXXX XXXX XXXX XXXX, IA XXXX To Whom It May Concern, I am writing to formally dispute unauthorized hard i


In [19]:
print(type(documents))
print(len(documents))
print(documents[0][:200])

<class 'list'>
2
XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX Apt XXXX, XXXX, TX XXXX XXXX : XX/XX/XXXX TransUnion Consumer Solutions XXXX XXXX XXXX XXXX, PA XXXX XXXX : XXXX Re : Security Freeze Request Dear Sir/Madam, I,


In [20]:
chunks = retrieve_chunks(
    "What are the common complaints?"
)

print(chunks[0]["text"][:500])

Subject : Dispute of Unauthorized Hard Inquiries on Credit Report XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX, Fl XXXX XX/XX/XXXX Social Security XXXX XX/XX/XXXX Consumer Financial Protection Bureau XXXX XXXX XXXX XXXX, IA XXXX To Whom It May Concern, I am writing to formally dispute unauthorized hard inquiries that have appeared on my credit report. I did not authorize these inquiries, and they are negatively impacting my credit score. Despite previous attempts to resolve this matter with the credi


In [21]:
def rag_pipeline(question):

    retrieved_chunks = retrieve_chunks(question)

    context = "\n\n".join(
        chunk["text"]
        for chunk in retrieved_chunks
    )

    prompt = f"""
Use the following context to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

    answer = generate_answer(prompt)

    return answer, retrieved_chunks

In [22]:
def retrieve_chunks(question, k=3):

    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    )

    distances, indices = index.search(
        query_embedding.astype(np.float32),
        k
    )

    retrieved = []

    for idx in indices[0]:
        retrieved.append(metadata[idx])

    return retrieved

In [23]:
print("embedding_model" in globals())
print("index" in globals())
print("metadata" in globals())
print("retrieve_chunks" in globals())

True
True
True
True


In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="distilgpt2"
)

print("Generator loaded successfully")

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

In [ ]:

import gradio as gr
import faiss
import numpy as np
import time

from sentence_transformers import SentenceTransformer
from transformers import pipeline

# ==========================================
# LOAD EMBEDDING MODEL
# ==========================================

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

# ==========================================
# LOAD GENERATION MODEL
# ==========================================

generator = pipeline(
    "text-generation",
    model="distilgpt2"
)

# ==========================================
# BUILD FAISS INDEX
# ==========================================

embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True,
    show_progress_bar=True
)

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    embeddings.astype(np.float32)
)

metadata = [
    {"text": doc}
    for doc in documents
]

# ==========================================
# RETRIEVAL FUNCTION
# ==========================================

def retrieve_chunks(question, k=3):

    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    )

    distances, indices = index.search(
        query_embedding.astype(np.float32),
        k
    )

    results = []

    for idx in indices[0]:
        results.append(metadata[idx])

    return results

# ==========================================
# ANSWER GENERATION
# ==========================================

def generate_answer(prompt):

    result = generator(
        prompt,
        max_new_tokens=150,
        do_sample=False,
        truncation=True
    )

    generated_text = result[0]["generated_text"]

    return generated_text.replace(prompt, "").strip()

# ==========================================
# RAG PIPELINE
# ==========================================

def rag_pipeline(question):

    retrieved_chunks = retrieve_chunks(
        question,
        k=3
    )

    context = "\n\n".join(
        chunk["text"]
        for chunk in retrieved_chunks
    )

    prompt = f"""
You are an assistant that answers questions using customer complaint records.

Complaint Records:
{context}

Question:
{question}

Provide a concise answer based only on the complaint records.
"""

    answer = generate_answer(prompt)

    return answer, retrieved_chunks

# ==========================================
# STREAMING
# ==========================================

def ask_question(question):

    answer, retrieved_chunks = rag_pipeline(question)

    sources = "\n\n".join(
        [
            f"Source {i+1}:\n{chunk['text'][:500]}"
            for i, chunk in enumerate(retrieved_chunks)
        ]
    )

    partial_answer = ""

    for word in answer.split():
        partial_answer += word + " "
        time.sleep(0.03)
        yield partial_answer, sources

# ==========================================
# CLEAR
# ==========================================

def clear_all():
    return "", "", ""

# ==========================================
# UI
# ==========================================

with gr.Blocks(title="Complaint RAG Chatbot") as demo:

    gr.Markdown("# Complaint RAG Chatbot")

    gr.Markdown(
        """
Ask questions about customer complaints.

Features:
- AI-generated answers
- Source transparency
- Streaming responses
- Clear button
        """
    )

    question_input = gr.Textbox(
        label="Question",
        placeholder="What are the common complaints about credit reporting?"
    )

    with gr.Row():
        ask_btn = gr.Button("Ask")
        clear_btn = gr.Button("Clear")

    answer_output = gr.Textbox(
        label="AI Generated Answer",
        lines=8
    )

    source_output = gr.Textbox(
        label="Retrieved Sources",
        lines=12
    )

    ask_btn.click(
        fn=ask_question,
        inputs=question_input,
        outputs=[
            answer_output,
            source_output
        ]
    )

    clear_btn.click(
        fn=clear_all,
        outputs=[
            question_input,
            answer_output,
            source_output
        ]
    )

demo.launch()


In [21]:
print(type(rag_pipeline))

<class 'function'>
